# Longitudinal guides: are they informative?

What the guided nets actually receive: for each sample, the current study's T1 and CT1 and the
**prior study's CT1 at slices z-2 .. z+2** (the w2 guide set), drawn by the **same loader the
training run uses** (its config, its mask, its slice matching), from the train or val split.

Three questions, one section each:

1. **Look at the guides.** Current T1 / CT1 next to the prior CT1 window, and the prior T1.
2. **Is there progression?** Enhancement now vs. then, and the signed CT1 change. Samples can be
   ranked by how much enhancement appeared or disappeared, so the interesting ones come first.
3. **Is the slice range right?** Where enhancement actually lives along z, over many volumes, with
   the current `SLICE_RANGE` marked. Change `SLICE_RANGE` in the next cell and re-run.

Everything is read-only.

In [ ]:
import os, sys, math
import numpy as np
import torch
import yaml
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

from visualization.image import set_display_orient, subplot_images
from visualization.hist import plot_hist
set_display_orient("radiological")            # display only; the stored pixels are RAS

In [ ]:
# ---- what to look at ------------------------------------------------------------------------------
CONFIG = "config/NYUMets/i2sb_sbguidedcdl_otherstudy_w2_sfree_flex_bs64.json"   # any guided config
SPLIT = "train"              # "train" | "val": that config's own data block for the split
SLICE_RANGE = (50, 90)       # ORIGINAL slice indices [lo, hi); None = every stored slice
GUIDE_WINDOW = 2             # prior slices z-W .. z+W (2 = the w2 guide set)
ROOT_OVERRIDE = None         # a different h5 root than the config's, e.g. another build

# ---- which samples ---------------------------------------------------------------------------------
N_CANDIDATES = 300           # random samples scored for change
N_SHOW = 6                   # rows drawn
SHOW = "top"                 # "top": most new / vanished enhancement first | "random"
SEED = 0
ENH_THR = 0.15               # (CT1 - T1)+ above this counts as enhancement (scaled units)
MASK_DISPLAY = True          # draw every panel inside its own row's T1 & CT1 mask

# ---- slice-range explorer ---------------------------------------------------------------------------
N_VOLUMES = 40               # volumes read whole for the per-slice enhancement profile

In [ ]:
import datasets                                   # noqa: F401  (registers the loaders)
from datasets.registry import build_loader

with open(CONFIG) as f:
    CFG = yaml.safe_load(f)
d = dict(CFG["data"][SPLIT])
X0, X1 = int(d.get("x0_idx", 2)), int(d.get("x1_idx", 1))
GUIDE_IDX = [X1, X0]                              # the prior T1 AND CT1 at every offset
d.update(name="nyumets_guided", guide_mode="other_study", guide_idx=GUIDE_IDX,
         guide_window=GUIDE_WINDOW, guide_enhancement=False, guide_as_cond=False, n_guides=1,
         x1_source="contrast", deterministic=True, random_flips=False, crop_size=None,
         center_crop=None, batch_size=1, num_workers=0,
         slice_range=None if SLICE_RANGE is None else list(SLICE_RANGE))
if ROOT_OVERRIDE:
    d["root"] = ROOT_OVERRIDE
DS = build_loader(d, shuffle=False, drop_last=False).dataset
NC, W = len(GUIDE_IDX), GUIDE_WINDOW
print("%s / %s: %d slices from %d volumes (multi-study patients only), root %s"
      % (os.path.basename(CONFIG), SPLIT, len(DS), len(DS.img_paths), d["root"]))
print("guide slice matching: %s | mask %s, applied=%s, to guides: %s | intensity_norm %s"
      % (DS.guide_slice, d.get("mask_source"), d.get("mask_apply"), d.get("mask_guides"),
         DS.intensity_norm))
print("NOTE: the training loader draws the sibling study at random (deterministic=False); "
      "this one fixes it per sample so a figure is reproducible.")

## Score candidates and pick rows

Per sample, inside the mask: `new` = share of pixels enhancing now but in **none** of the prior
slices z-W..z+W (lenient to a residual slice offset), `gone` = the reverse, `chg` = RMS of
CT1 - prior CT1(z) relative to RMS of CT1, `corr` = correlation of CT1 with prior CT1(z) (a
registration check: low means the guide is not the same anatomy).

"Prior" is just the OTHER study the loader matched: the studies carry no dates, so it may be
earlier or later. `new` and `gone` are therefore both "change"; read them together.

In [ ]:
def plane(g, d, c):
    """(G, H, W) guide stack -> the prior plane at offset d of stored contrast c."""
    return g[(W + d) * NC + GUIDE_IDX.index(c)]


def masked_corr(a, b, m):
    a, b = a[m], b[m]
    if a.numel() < 50:
        return float("nan")
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm()).clamp_min(1e-8))


def load(i):
    """Sample i -> dict of (H, W) tensors and its scores."""
    s = DS[i]
    x0, x1, mask, g = s["x0"][0], s["x1"][0], s["mask"][0] > 0.5, s["guide"][:, 0]
    oct1 = torch.stack([plane(g, k, X0) for k in range(-W, W + 1)])       # (2W+1, H, W)
    ot1 = torch.stack([plane(g, k, X1) for k in range(-W, W + 1)])
    s_now = (x0 - x1).clamp_min(0)
    s_pri = (oct1 - ot1).clamp_min(0)                                      # per offset
    e_now = (s_now > ENH_THR) & mask
    e_pri = (s_pri > ENH_THR).any(0) & mask                                 # any prior offset
    n = max(int(mask.sum()), 1)
    c = oct1[W]
    rms = lambda t: float(t[mask].pow(2).mean().sqrt()) if mask.any() else float("nan")
    fi, li = int(DS.file_id[i]), int(DS.local[i])
    return dict(i=i, x0=x0, x1=x1, mask=mask.float(), oct1=oct1, ot1=ot1, s_now=s_now, s_pri=s_pri,
                z=int(DS._slice_index(fi)[li]),
                new=float((e_now & ~e_pri).sum()) / n, gone=float((e_pri & ~e_now).sum()) / n,
                enh=float(e_now.sum()) / n,
                chg=rms(x0 - c) / max(rms(x0), 1e-8), corr=masked_corr(x0, c, mask))


def draw(seed):
    """-> (SAMPLES, ROWS): N_CANDIDATES random samples scored, and the N_SHOW to draw."""
    rng = np.random.default_rng(seed)
    cand = rng.choice(len(DS), size=min(N_CANDIDATES, len(DS)), replace=False)
    samples = [load(int(i)) for i in cand]
    if SHOW == "top":
        rows = sorted(samples, key=lambda s: -(s["new"] + s["gone"]))[:N_SHOW]
    else:
        rows = samples[:N_SHOW]
    return samples, rows


def table(rows):
    print("%5s %6s %7s %7s %7s %6s %6s" % ("row", "slice", "enh now", "new", "gone", "chg", "corr"))
    for r, s in enumerate(rows):
        print("%5d %6d %7.3f %7.3f %7.3f %6.2f %6.2f"
              % (r, s["z"], s["enh"], s["new"], s["gone"], s["chg"], s["corr"]))


SAMPLES, ROWS = draw(SEED)
table(ROWS)

## 1. The guides

Gray, one window for every panel (from the current CT1s). Prior columns are the prior study's CT1
at z-2 .. z+2 as the loader matched them, then its T1 at z.

In [ ]:
def mrow(s, t):
    return t * s["mask"] if MASK_DISPLAY else t


def show_guides(rows_):
    offs = list(range(-W, W + 1))
    cols = (["T1 now", "CT1 now"] + ["prior CT1 z" if k == 0 else "prior CT1 z%+d" % k for k in offs]
            + ["prior T1 z"])
    rows = [[mrow(s, s["x1"]), mrow(s, s["x0"])] + [mrow(s, s["oct1"][W + k]) for k in offs]
            + [mrow(s, s["ot1"][W])] for s in rows_]
    subplot_images(rows, row_labels=["slice %d" % s["z"] for s in rows_], col_titles=cols,
                   cmap="gray", window_from=[torch.stack([s["x0"] for s in rows_])], p=(1, 99),
                   magnitude=False, panel_size=(2.0, 2.2), show=False)
    plt.show()


show_guides(ROWS)

## 2. Is there progression?

`S now` and `S prior` are the enhancement maps (CT1 - T1)+ of the two studies at the matched slice,
on one window. `S now - S prior`: red = enhancement that appeared, blue = enhancement that went
away. `CT1 - prior CT1`: the change a longitudinal net has to explain.

In [ ]:
def _q(ts, q):
    v = torch.cat([t.flatten() for t in ts])
    return max(float(torch.quantile(v[::max(1, v.numel() // 1_000_000)], q)), 1e-3) if v.numel() else 1.0


def show_change(rows_):
    vS = _q([s["s_now"][s["mask"] > 0] for s in rows_], 0.995)
    vD = _q([(s["x0"] - s["oct1"][W])[s["mask"] > 0].abs() for s in rows_], 0.99)
    rows = [[mrow(s, s["s_now"]), mrow(s, s["s_pri"][W]), mrow(s, s["s_now"] - s["s_pri"][W]),
             mrow(s, s["x0"] - s["oct1"][W])] for s in rows_]
    subplot_images(rows, row_labels=["slice %d" % s["z"] for s in rows_],
                   col_titles=["S now", "S prior (z)", "S now - S prior", "CT1 - prior CT1"],
                   cmap=["magma", "magma", "RdBu_r", "RdBu_r"],
                   vmin=[0.0, 0.0, -vS, -vD], vmax=[vS, vS, vS, vD], share_window=False,
                   magnitude=False, panel_size=(2.4, 2.6), colorbar="each", show=False)
    plt.show()


show_change(ROWS)

In [ ]:
def show_hists(samples):
    """How often there is any change at all, over every scored candidate."""
    plot_hist({"new enhancement": np.array([s["new"] for s in samples]),
               "vanished enhancement": np.array([s["gone"] for s in samples])},
              bins=40, xlabel="share of brain pixels", title="enhancement change per sample "
              "(%d samples, %s, slices %s)" % (len(samples), SPLIT, SLICE_RANGE), show=True)
    plot_hist(np.array([s["corr"] for s in samples]), bins=40, xlabel="corr(CT1, prior CT1 at z)",
              title="is the matched prior slice the same anatomy?", show=True)


show_hists(SAMPLES)

### Are the two studies on one intensity scale?

Median inside the mask, current over prior, for the CT1 and the T1 of every scored candidate. With
the config's `intensity_norm: volume_median` both should sit near 1; with fixed `scales` the
spread is each scan's own gain.

In [ ]:
def show_scale(samples):
    med = lambda t, m: float(t[m > 0].median()) if bool((m > 0).any()) else float("nan")
    r_ct1 = np.array([med(s["x0"], s["mask"]) / max(med(s["oct1"][W], s["mask"]), 1e-6)
                      for s in samples])
    r_t1 = np.array([med(s["x1"], s["mask"]) / max(med(s["ot1"][W], s["mask"]), 1e-6)
                     for s in samples])
    for name, r in (("CT1", r_ct1), ("T1 ", r_t1)):
        r = r[np.isfinite(r)]
        print("%s now / prior, median in the mask: p10 %.2f  p50 %.2f  p90 %.2f   (%s)"
              % (name, *np.percentile(r, [10, 50, 90]), DS.intensity_norm))


show_scale(SAMPLES)

## Refresh

New candidates with the next seed (or a given one), then the table and the figures of sections 1
and 2. `refresh(hists=True)` redraws the histograms too. Settings changed in the second cell
(`SHOW`, `N_SHOW`, `ENH_THR`, `MASK_DISPLAY`) apply on the next refresh; `SPLIT`, `SLICE_RANGE` and
`GUIDE_WINDOW` change the loader, so re-run from the loader cell for those.

In [ ]:
_seed = SEED


def refresh(seed=None, hists=False):
    global SAMPLES, ROWS, _seed
    _seed = _seed + 1 if seed is None else int(seed)
    SAMPLES, ROWS = draw(_seed)
    print("seed %d | %d candidates scored | showing %d (%s)" % (_seed, len(SAMPLES), len(ROWS), SHOW))
    table(ROWS)
    show_guides(ROWS)
    show_change(ROWS)
    if hists:
        show_hists(SAMPLES)


refresh()

## 3. Is the slice range right?

Whole volumes, every stored slice: the share of brain pixels enhancing, per **original** slice index,
averaged over volumes (band = 25-75%), and the brain area per slice. Enhancement outside the dashed
lines is never seen in training.

In [ ]:
key_img = d.get("image_key", "img_raw")
scales = None if d.get("scales") is None else np.asarray(d["scales"], dtype=np.float32)
eps = float(d.get("mask_eps", 0.0) or 0.0)
pick = np.random.default_rng(SEED).choice(len(DS.img_paths), size=min(N_VOLUMES, len(DS.img_paths)),
                                          replace=False)
per_z, area_z = {}, {}
for fi in pick:
    with h5py.File(DS.img_paths[fi], "r") as f:
        v = np.asarray(f[key_img][:, :, :, [X1, X0]], dtype=np.float32)
        zs = np.asarray(f["slice_index"])
        centre = (np.asarray(f.attrs["norm_stats"], dtype=np.float32).reshape(-1, 2)[[X1, X0], 0]
                  if d.get("intensity_norm") == "volume_median" else None)
    if centre is not None:
        v = v / centre                      # the loader's intensity_norm, as in training
    elif scales is not None:
        v = v / scales[[X1, X0]]
    m = (np.abs(v) > eps).all(-1)
    s = np.clip(v[..., 1] - v[..., 0], 0, None)
    for z, mz, sz in zip(zs, m, s):
        n = mz.sum()
        if n < 200:
            continue
        per_z.setdefault(int(z), []).append(float(((sz > ENH_THR) & mz).sum()) / n)
        area_z.setdefault(int(z), []).append(float(mz.mean()))

zz = np.array(sorted(per_z))
q = lambda dct, p: np.array([np.percentile(dct[z], p) for z in zz])
fig, ax = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True)
ax[0].plot(zz, [np.mean(per_z[z]) for z in zz], label="mean")
ax[0].fill_between(zz, q(per_z, 25), q(per_z, 75), alpha=0.3, label="25-75%")
ax[0].set_ylabel("enhancing share of brain")
ax[1].plot(zz, [np.mean(area_z[z]) for z in zz])
ax[1].set_ylabel("brain area share")
ax[1].set_xlabel("original slice index")
for a in ax:
    if SLICE_RANGE is not None:
        for b in SLICE_RANGE:
            a.axvline(b, ls="--", color="k", lw=1)
    a.grid(alpha=0.3)
ax[0].legend()
ax[0].set_title("%d volumes from %s; dashed = SLICE_RANGE %s" % (len(pick), SPLIT, SLICE_RANGE))
plt.tight_layout()
plt.show()
inside = sum(sum(per_z[z]) for z in zz if SLICE_RANGE and SLICE_RANGE[0] <= z < SLICE_RANGE[1])
print("share of all enhancing area (summed over slices) that SLICE_RANGE keeps: %.0f%%"
      % (100 * inside / max(sum(sum(v) for v in per_z.values()), 1e-12)))